# 面包店数据探索性分析

运行前请确保已 `pip install -r requirements.txt`，并把数据放到 `data/raw/food.xlsx`（不存在时自动用样例数据）。

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src import config, load, clean, analysis

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 30)

frames = load.load_all()
frames = clean.clean_all(frames)
print("数据源:", frames["_source"].iloc[0, 0])
for name, df in frames.items():
    if not name.startswith("_"):
        print(f"{name}: {len(df)} 行")

## 1. 销售概览

In [ ]:
sales = frames[config.SHEET_DAILY_SALES]
purchases = frames[config.SHEET_PURCHASE]
analysis.sales_kpis(purchases, sales)

## 2. 产品与口味排名

In [ ]:
product_rank = analysis.product_ranking(sales)
display(product_rank)

taste_rank = analysis.taste_ranking(sales, frames[config.SHEET_BOM])
display(taste_rank)

## 3. 产品销量对比图

In [ ]:
import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(product_rank["商品名称"], product_rank["总销量"], color="#C97B63")
ax.set_title("各产品总销量")
ax.set_ylabel("销量（件）")
plt.tight_layout()
plt.show()

## 4. 节假日效应与客流

In [ ]:
display(analysis.holiday_effect(sales))
display(analysis.traffic_analysis(frames[config.SHEET_TRAFFIC]).head(10))

## 5. 生产与销售匹配、原料消耗

In [ ]:
display(analysis.production_vs_sales(frames[config.SHEET_PRODUCTION], sales).head(10))
display(analysis.material_consumption(frames[config.SHEET_CONSUMPTION], frames[config.SHEET_MATERIAL]))